# Orthogonal Procrustes Drift Pipeline

This notebook:
- computes Orthogonal Procrustes transforms between consecutive decades,
- measures pairwise semantic drift from aligned embeddings,
- composes transforms toward a reference decade,
- tracks long-term drift trajectories for selected words.

## Pipeline Overview

### Inputs
- Shared vocabulary (`VOCAB_FILE`) mapping each word to a stable index.
- Decade-level frequency dictionaries from `FREQUENCIES_DIR`.
- Trained embedding matrices (`emb_<decade>.pt`) from `MODELS_DIR`.

### Stage 1: Pairwise Alignment and Local Drift
1. Select anchor words shared by two consecutive decades with a minimum frequency threshold.
2. Build anchor matrices `X` and `Y` using the common vocabulary index mapping.
3. Center anchors and estimate the Orthogonal Procrustes rotation matrix.
4. Align embeddings from decade `t` into decade `t+1` space.
5. Compute local drift (`1 - cosine`) and keep top-K drifting words among valid non-anchor words.

### Stage 2: Global Drift to a Reference
1. Load saved pairwise transforms.
2. Compose them forward from each decade to the reference decade.
3. Align target word vectors into the reference space.
4. Save per-decade trajectories of cosine similarity and drift.

In [2]:
import json
from pathlib import Path
import numpy as np
import torch

from src.config import (
    DECADES,
    VOCAB_FILE_EXPANDED,
    FREQUENCIES_DIR_EXPANDED,
    MODELS_DIR,
    ALIGNMENT_TRANSFORMS_DIR_EXPANDED,
    ALIGNMENT_DRIFT_RESULTS_DIR_EXPANDED,
    ALIGNMENT_GLOBAL_RESULTS_DIR_EXPANDED,
)

Path(ALIGNMENT_TRANSFORMS_DIR_EXPANDED).mkdir(parents=True, exist_ok=True)
Path(ALIGNMENT_DRIFT_RESULTS_DIR_EXPANDED).mkdir(parents=True, exist_ok=True)
Path(ALIGNMENT_GLOBAL_RESULTS_DIR_EXPANDED).mkdir(parents=True, exist_ok=True)

print('Decades:', len(DECADES), DECADES[0], '->', DECADES[-1])
print('VOCAB_FILE:', VOCAB_FILE_EXPANDED)
print('FREQUENCIES_DIR:', FREQUENCIES_DIR_EXPANDED)
print('MODELS_DIR:', MODELS_DIR)
print('ALIGNMENT_TRANSFORMS_DIR:', ALIGNMENT_TRANSFORMS_DIR_EXPANDED)
print('ALIGNMENT_DRIFT_RESULTS_DIR:', ALIGNMENT_DRIFT_RESULTS_DIR_EXPANDED)
print('ALIGNMENT_GLOBAL_RESULTS_DIR:', ALIGNMENT_GLOBAL_RESULTS_DIR_EXPANDED)


Decades: 12 1900s -> 2010s
VOCAB_FILE: /home/ccoppola/projects/test/diachronic_text_analysis_camilla/data/processed/5gram-expanded/vocab.json
FREQUENCIES_DIR: /home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/frequencies
MODELS_DIR: /home/ccoppola/projects/test/diachronic_text_analysis_camilla/models/5gram-full
ALIGNMENT_TRANSFORMS_DIR: /home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/transforms
ALIGNMENT_DRIFT_RESULTS_DIR: /home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/drift_results
ALIGNMENT_GLOBAL_RESULTS_DIR: /home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/global_results


In [3]:
# Runtime parameters
MIN_FLOOR = 50
MIN_COUNT_RATIO = 1e-5
MIN_DRIFT_COUNT = 190
TOP_K = 50
REFERENCE_DECADE = '2010s'
TARGET_WORDS = ['computer', 'internet', 'cloud']

print('MIN_FLOOR =', MIN_FLOOR)
print('MIN_COUNT_RATIO =', MIN_COUNT_RATIO)
print('MIN_DRIFT_COUNT =', MIN_DRIFT_COUNT)
print('TOP_K =', TOP_K)
print('REFERENCE_DECADE =', REFERENCE_DECADE)
print('TARGET_WORDS =', TARGET_WORDS)


MIN_FLOOR = 50
MIN_COUNT_RATIO = 1e-05
MIN_DRIFT_COUNT = 190
TOP_K = 50
REFERENCE_DECADE = 2010s
TARGET_WORDS = ['computer', 'internet', 'cloud']


In [4]:
# Helper functions 
def load_common_vocab(vocab_file=VOCAB_FILE_EXPANDED):
    with open(vocab_file, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_frequencies(decade, freq_dir=FREQUENCIES_DIR_EXPANDED):
    freq_file = Path(freq_dir) / f'freq_{decade}.json'

    if not freq_file.exists():
        raise FileNotFoundError(f'Frequency file not found: {freq_file}')
    
    with open(freq_file, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_all_frequencies(decades):
    return {d: load_frequencies(d) for d in decades}

def load_embedding_matrix_pt(decade, models_dir=MODELS_DIR):
    pt_path = Path(models_dir) / f'emb_{decade}.pt'
    if not pt_path.exists():
        raise FileNotFoundError(f'Missing embedding file: {pt_path}')
    
    state = torch.load(pt_path, map_location='cpu')

    key = 'embeddings.weight'
    if key not in state:
        raise KeyError(
                    f"Key '{key}' not found in {pt_path}.\n"
                    f"Available: {list(state.keys())}"
        )
    
    return state[key].detach().cpu().numpy()

def compute_global_min_count(freq_by_decade, ratio=MIN_COUNT_RATIO, min_floor=MIN_FLOOR):
    total_tokens = 0

    for freq in freq_by_decade.values():
        total_tokens += sum(freq.values())

    avg_tokens = total_tokens / len(freq_by_decade)
    return max(min_floor, int(ratio * avg_tokens))


def select_anchor_words(
    freq_t: dict,
    freq_t1: dict,
    min_count: int,
    max_freq_ratio: float = 2.0
) :
   
    shared_words = set(freq_t.keys()) & set(freq_t1.keys())
    anchors = []

    for word in shared_words:
        c_t = freq_t[word]
        c_t1 = freq_t1[word]

        if c_t < min_count or c_t1 < min_count:
            continue

        ratio = max(c_t, c_t1) / max(1, min(c_t, c_t1))
        if ratio <= max_freq_ratio:
            anchors.append(word)

    return sorted(anchors)

def build_alignment_matrices(anchor_words, E_t, E_t1, vocab):
    X_rows, Y_rows, used_anchors = [], [], []

    for w in anchor_words:
        idx = vocab.get(w)
        if idx is None:
            print(f'Skipping anchor word "{w}": not in vocab.')
            continue
        if idx >= E_t.shape[0] or idx >= E_t1.shape[0]:
            print(f'Skipping anchor word "{w}": index {idx} out of range for embedding matrices.')
            continue

        X_rows.append(E_t[idx])
        Y_rows.append(E_t1[idx])
        used_anchors.append(w)

    if not X_rows or not Y_rows:
        raise ValueError('No valid anchor words available for alignment matrices.')
    
    X = np.vstack(X_rows) # shape: (|anchors|, d)
    Y = np.vstack(Y_rows) # shape: (|anchors|, d)
    return X, Y, used_anchors

def orthogonal_procrustes(X, Y):
    M = X.T @ Y
    U, _, Vt = np.linalg.svd(M)
    R = U @ Vt
    return R

def cosine_similarity(u, v):
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

def cosine_all_rows(A, B):
    num = np.sum(A * B, axis=1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1)
    return num / den

def get_valid_drift_indices(vocab, freq_t, freq_t1, used_anchors, min_drift_count, V):
    mask = np.ones(V, dtype=bool)

    anchor_idx = np.array(
        [vocab[w] for w in used_anchors if w in vocab], 
        dtype=int
    )
    if anchor_idx.size > 0:
        mask[anchor_idx] = False

    valid_idx = []
    for w, idx in vocab.items():
        if idx >= V:
            continue
        if not mask[idx]:
            continue
        if freq_t.get(w, 0) >= min_drift_count and freq_t1.get(w, 0) >= min_drift_count:
            valid_idx.append(idx)
    return np.array(valid_idx, dtype=int)

def save_alignment_pair(d_t, d_t1, R, mu_X, mu_Y, n_anchors, min_count):
    obj = {
        'from': d_t,
        'to': d_t1,
        'R': torch.from_numpy(R).float(),
        'mu_X': torch.from_numpy(mu_X).float(),
        'mu_Y': torch.from_numpy(mu_Y).float(),
        'n_anchors': n_anchors,
        'min_count': min_count,
    }
    out_path = Path(ALIGNMENT_TRANSFORMS_DIR_EXPANDED) / f'alignment_{d_t}_to_{d_t1}.pt'
    torch.save(obj, out_path)
    return out_path

def load_alignment_pair(d_t, d_t1):
    path = Path(ALIGNMENT_TRANSFORMS_DIR_EXPANDED) / f'alignment_{d_t}_to_{d_t1}.pt'
    obj = torch.load(path, map_location='cpu')
    return {
        'R': obj['R'].numpy(),
        'mu_X': obj['mu_X'].numpy(),
        'mu_Y': obj['mu_Y'].numpy(),
    }

def apply_pair_transform(E, R, mu_X, mu_Y):
    return (E - mu_X) @ R + mu_Y

def align_decade_to_reference(decade, ref_decade, E_dec):
    if decade == ref_decade:
        return E_dec
    
    i = DECADES.index(decade)
    j = DECADES.index(ref_decade)
    if i > j:
        raise ValueError('This forward composition only supports decade <= ref_decade.')
    E = E_dec
    for k in range(i, j):
        d_t = DECADES[k]
        d_t1 = DECADES[k + 1]
        T = load_alignment_pair(d_t, d_t1)
        E = apply_pair_transform(E, T['R'], T['mu_X'], T['mu_Y'])
    return E


## Part A - Pairwise Procrustes Alignment and Local Drift

What happens in this stage:
- anchor selection based on shared frequency support,
- estimation of one orthogonal transform per decade pair,
- alignment of source embeddings into target space,
- extraction of top-K local drift terms after filtering anchors and low-frequency words.

Interpretation notes:
- a larger drift score (`1 - cosine`) means stronger semantic movement between adjacent decades,
- anchors are excluded from final drift ranking to reduce trivial high-frequency stability effects,
- quality depends on vocabulary consistency and embedding row/index alignment.

In [5]:
vocab = load_common_vocab(VOCAB_FILE_EXPANDED)
freq_by_decade = load_all_frequencies(DECADES)
global_min_count = compute_global_min_count(freq_by_decade)
idx2word = {i: w for w, i in vocab.items()}

print('Vocabulary size:', len(vocab))
print('Global anchor min_count:', global_min_count)


Vocabulary size: 50002
Global anchor min_count: 1334


In [6]:
pairwise_summary = []

for i in range(len(DECADES) - 1):
    d_t = DECADES[i]
    d_t1 = DECADES[i + 1]

    E_t = load_embedding_matrix_pt(d_t)
    E_t1 = load_embedding_matrix_pt(d_t1)

    anchors = select_anchor_words(freq_by_decade[d_t], freq_by_decade[d_t1], min_count=global_min_count)
    X, Y, used_anchors = build_alignment_matrices(anchors, E_t, E_t1, vocab)

    mu_X = X.mean(axis=0, keepdims=True)
    mu_Y = Y.mean(axis=0, keepdims=True)
    Xc = X - mu_X
    Yc = Y - mu_Y

    R = orthogonal_procrustes(Xc, Yc)
    out_transform = save_alignment_pair(d_t, d_t1, R, mu_X, mu_Y, len(used_anchors), global_min_count)

    E_t_aligned = apply_pair_transform(E_t, R, mu_X, mu_Y)
    cos = cosine_all_rows(E_t_aligned, E_t1)
    drift = 1.0 - cos

    valid_idx = get_valid_drift_indices(
        vocab=vocab,
        freq_t=freq_by_decade[d_t],
        freq_t1=freq_by_decade[d_t1],
        used_anchors=used_anchors,
        min_drift_count=MIN_DRIFT_COUNT,
        V=E_t1.shape[0],
    )

    if len(valid_idx) > 0:
        idx_sorted = np.argsort(drift[valid_idx])[::-1]
        top_idx = valid_idx[idx_sorted[:TOP_K]]
    else:
        top_idx = np.array([], dtype=int)

    out_data = [
        {
            'word': idx2word[idx],
            'drift': float(drift[idx]),
            'cosine': float(cos[idx]),
            'from': d_t,
            'to': d_t1,
        }
        for idx in top_idx
    ]

    out_drift = Path(ALIGNMENT_DRIFT_RESULTS_DIR_EXPANDED) / f'drift_{d_t}_to_{d_t1}.json'
    with open(out_drift, 'w', encoding='utf-8') as f:
        json.dump(out_data, f, ensure_ascii=False, indent=2)

    pairwise_summary.append({
        'pair': f'{d_t}->{d_t1}',
        'anchors_requested': len(anchors),
        'anchors_used': len(used_anchors),
        'valid_drift_words': len(valid_idx),
        'transform_file': str(out_transform),
        'drift_file': str(out_drift),
    })

print('Pairwise stage completed for', len(pairwise_summary), 'decade pairs.')


Pairwise stage completed for 11 decade pairs.


In [7]:
for row in pairwise_summary[:5]:
    print(row)
if len(pairwise_summary) > 5:
    print('...')


{'pair': '1900s->1910s', 'anchors_requested': 3558, 'anchors_used': 3558, 'valid_drift_words': 7719, 'transform_file': '/home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/transforms/alignment_1900s_to_1910s.pt', 'drift_file': '/home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/drift_results/drift_1900s_to_1910s.json'}
{'pair': '1910s->1920s', 'anchors_requested': 3494, 'anchors_used': 3494, 'valid_drift_words': 7364, 'transform_file': '/home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/transforms/alignment_1910s_to_1920s.pt', 'drift_file': '/home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/drift_results/drift_1910s_to_1920s.json'}
{'pair': '1920s->1930s', 'anchors_requested': 3276, 'anchors_used': 3276, 'valid_drift_words': 6804, 'transform_file': '/home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/transforms/alignment_1920s_to

## Sanity Checks (Recommended Before Part B)

This diagnostic block mirrors the most useful checks.

Why this is important:
- It verifies that the estimated transform is numerically orthogonal.
- It confirms that alignment reduces the anchor-space mismatch.
- It checks that cosine similarity improves on a known test word.
- It measures average cosine gain on a sample of anchor words.

In [8]:
# Sanity-check parameters (read-only diagnostics)
SANITY_PAIR = (DECADES[-2], DECADES[-1])
SANITY_WORD = 'computer'
SANITY_ANCHOR_SAMPLE = 50

d_t_chk, d_t1_chk = SANITY_PAIR
print('Running sanity checks for pair:', d_t_chk, '->', d_t1_chk)

E_t_chk = load_embedding_matrix_pt(d_t_chk)
E_t1_chk = load_embedding_matrix_pt(d_t1_chk)

print('E_t shape:', E_t_chk.shape)
print('E_t1 shape:', E_t1_chk.shape)
print('Vocab size:', len(vocab))
if E_t_chk.shape[0] != len(vocab) or E_t1_chk.shape[0] != len(vocab):
    print('WARNING: embedding rows and vocab size are not aligned.')

anchors_chk = select_anchor_words(
    freq_by_decade[d_t_chk],
    freq_by_decade[d_t1_chk],
    min_count=global_min_count,
    )
X_chk, Y_chk, used_anchors_chk = build_alignment_matrices(anchors_chk, E_t_chk, E_t1_chk, vocab)

mu_X_chk = X_chk.mean(axis=0, keepdims=True)
mu_Y_chk = Y_chk.mean(axis=0, keepdims=True)
Xc_chk = X_chk - mu_X_chk
Yc_chk = Y_chk - mu_Y_chk
R_chk = orthogonal_procrustes(Xc_chk, Yc_chk)

# Check orthogonality: R @ R^T should be identity (within numerical tolerance)
is_orthogonal = np.allclose(R_chk @ R_chk.T, np.eye(R_chk.shape[0]), atol=1e-6)
print(f"Is orthogonal (within 1e-6 tolerance)? {is_orthogonal}")

# Compute Frobenius norm of deviation from orthogonality
orthog_error = np.linalg.norm(R_chk @ R_chk.T - np.eye(R_chk.shape[0]))
print('Orthogonality error:', f'{orthog_error:.6e}')

# Compute Frobenius errors before and after alignment
before = np.linalg.norm(Yc_chk - Xc_chk, ord='fro')
after = np.linalg.norm(Yc_chk - Xc_chk @ R_chk, ord='fro')
after_t = np.linalg.norm(Yc_chk - Xc_chk @ R_chk.T, ord='fro')
print(f"Frobenius error (no alignment): {before:.6f}")
print(f"Frobenius error (X @ R):        {after:.6f}")
print(f"Frobenius error (X @ R.T):      {after_t:.6f}\n\n")

E_t_aligned_chk = apply_pair_transform(E_t_chk, R_chk, mu_X_chk, mu_Y_chk)

if SANITY_WORD in vocab:
    w_idx = vocab[SANITY_WORD]
    sim_before = cosine_similarity(E_t_chk[w_idx], E_t1_chk[w_idx])
    sim_after = cosine_similarity(E_t_aligned_chk[w_idx], E_t1_chk[w_idx])
    print(f"{SANITY_WORD} cosine before: {sim_before:.6f}")
    print(f"{SANITY_WORD} cosine after:  {sim_after:.6f}")
    print(f"{SANITY_WORD} delta:         {sim_after - sim_before:+.6f}")
else:
    print(f"{SANITY_WORD} not found in vocabulary")

sample_words = used_anchors_chk[:SANITY_ANCHOR_SAMPLE]
if sample_words:
    deltas = []
    for w in sample_words:
        idx = vocab[w]
        b = cosine_similarity(E_t_chk[idx], E_t1_chk[idx])
        a = cosine_similarity(E_t_aligned_chk[idx], E_t1_chk[idx])
        deltas.append(a - b)
    print('Anchor sample size:', len(sample_words))
    print('Mean anchor cosine delta (after-before):', f'{float(np.mean(deltas)):+.6f}')
else:
    print('No anchors available for sample-based sanity check.')

Running sanity checks for pair: 2000s -> 2010s
E_t shape: (50002, 300)
E_t1 shape: (50002, 300)
Vocab size: 50002
Is orthogonal (within 1e-6 tolerance)? True
Orthogonality error: 9.221789e-06
Frobenius error (no alignment): 1430.072754
Frobenius error (X @ R):        966.101685
Frobenius error (X @ R.T):      1428.098633


computer cosine before: 0.035770
computer cosine after:  0.540579
computer delta:         +0.504810
Anchor sample size: 50
Mean anchor cosine delta (after-before): +0.580039


## Part B - Global Drift to a Reference Decade

Conceptually:
- pairwise transforms define local maps (`t -> t+1`),
- chaining those maps projects older decades into a common reference space,
- once all vectors are in the same space, drift is comparable across the full timeline.

For each target word:
1. Load embedding vectors for every decade.
2. Align each decade to `REFERENCE_DECADE` via composed transforms.
3. Compute cosine similarity with the reference vector.
4. Store a time trajectory of `cosine_to_2010s` and `drift_to_2010s`.

Interpretation notes:
- cosine near 1.0 indicates strong semantic stability relative to the reference decade,
- larger drift indicates stronger long-term shift,
- trajectory shape (not just final value) helps identify when the major change occurred.

In [9]:
E_ref = load_embedding_matrix_pt(REFERENCE_DECADE)
global_files = []

for target_word in TARGET_WORDS:
    idx = vocab.get(target_word)
    if idx is None:
        print(f"Skipping '{target_word}': not in vocab")
        continue

    trajectory = []
    for d in DECADES:
        E_d = load_embedding_matrix_pt(d)
        E_d_aligned = align_decade_to_reference(d, REFERENCE_DECADE, E_d)
        sim = cosine_similarity(E_d_aligned[idx], E_ref[idx])
        drift = 1.0 - sim
        trajectory.append({
            'decade': d,
            'cosine_to_2010s': sim,
            'drift_to_2010s': drift,
        })

    out_file = Path(ALIGNMENT_GLOBAL_RESULTS_DIR_EXPANDED) / f'oTrajectory_{target_word}_to_{REFERENCE_DECADE}.json'
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(trajectory, f, ensure_ascii=False, indent=2)

    global_files.append(str(out_file))
    print('Saved:', out_file.name)

print('Global stage completed. Files:', len(global_files))


Saved: oTrajectory_computer_to_2010s.json
Saved: oTrajectory_internet_to_2010s.json
Saved: oTrajectory_cloud_to_2010s.json
Global stage completed. Files: 3


In [10]:
print('Pairwise outputs directory:')
print(Path(ALIGNMENT_DRIFT_RESULTS_DIR_EXPANDED))
print('\nGlobal outputs directory:')
print(Path(ALIGNMENT_GLOBAL_RESULTS_DIR_EXPANDED))

print('\nExample pairwise files:')
for p in sorted(Path(ALIGNMENT_DRIFT_RESULTS_DIR_EXPANDED).glob('drift_*.json'))[:3]:
    print('-', p.name)

print('\nExample global files:')
for p in sorted(Path(ALIGNMENT_GLOBAL_RESULTS_DIR_EXPANDED).glob('oTrajectory_*_to_*.json'))[:3]:
    print('-', p.name)


Pairwise outputs directory:
/home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/drift_results

Global outputs directory:
/home/ccoppola/projects/test/diachronic_text_analysis_camilla/alignment/5gram-full/global_results

Example pairwise files:
- drift_1900s_to_1910s.json
- drift_1910s_to_1920s.json
- drift_1920s_to_1930s.json

Example global files:
- oTrajectory_cloud_to_2010s.json
- oTrajectory_computer_to_2010s.json
- oTrajectory_internet_to_2010s.json
